# Batch 3D Soma Segmentation

Runs the soma segmentation from `Soma_Segmentation_3D.ipynb` over every `.ims` file in a folder and collects one table of soma sizes. The functions live in `NeuronAnnotation/Soma_Segmentation_functions.py`; use `Soma_Segmentation_3D.ipynb` to explore or tune a single file step by step.

**To use:** set the folder in the Settings cell, optionally test one file, then run the batch.

**Per file**, in the output folder:
- `<name>_soma_labels.tif`: refined label volume (open in Fiji/napari; voxel size is stored in the file)
- `<name>_soma_stats.csv`: one row per soma
- `<name>_candidates.csv`: every detected candidate and which filter it passed or failed
- `<name>_qc.png`: max projection with soma IDs and outlines (red = detected, yellow = refined)

**For the whole run:** `all_soma_stats.csv` (every soma), `run_log.csv` (ok / skipped / error per file) and `params.json` (the exact parameters used).

Voxel size is read from each file (Z varies between acquisitions). Single-plane 2D images and unreadable files are skipped and listed in the run log. Runs in the `tiffprocessing` env.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add the NeuronAnnotation folder to the Python path (notebook runs from the Notebooks folder)
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "NeuronAnnotation"))

import Soma_Segmentation_functions as soma

## Settings

Only `folder` is required. Any parameter left out of `params` uses the default from `soma.DEFAULT_PARAMS` (printed below), which match `Soma_Segmentation_3D.ipynb`.

In [ ]:
# --- Input / output ---
folder = r"D:\data\GFP"   # folder with the .ims files
out_dir = r"D:\data\GFP\soma_segmentation_v2"  # None = <folder>\soma_segmentation
pattern = "*.ims"           # e.g. "PS8292*.ims" for one strain only
skip_existing = True        # True = resume: files already processed are read back, not redone

# --- Parameters (override any default here) ---
params = {
    "boundary_fraction": 0.25,          # edge tightness: higher = smaller labels
    "soma_radius_um": 4.0,              # expected soma radius (sets process removal + soma spacing)
    "min_peak_separation_z_um": 8.0,    # soma centres closer than this in Z are one soma
    # brightness, as multiples of each image's own threshold:
    "min_peak_intensity_factor": 3.7,   # bright core: the main neuron-vs-background test
    "min_mean_intensity_factor": 1.3,
    # judged on the REFINED (trimmed) soma shape:
    "min_soma_volume_um3": 30.0,
    "max_soma_volume_um3": 2000.0,
    "min_solidity": 0.0,                # 0 = off (neurons here are elongated)
    "min_split_piece_um3": 20.0,        # two bright cores at least this big = two somas
}

resolved = soma.resolve_params(params)
out_dir = Path(out_dir) if out_dir else Path(folder) / "soma_segmentation"
print(f"{len(sorted(Path(folder).glob(pattern)))} file(s) match {pattern!r} in {folder}")
print(f"Output -> {out_dir}\n")
pd.Series(resolved, name="value").to_frame()

## Optional: test one file first

Takes about 1.5 min. Check the QC image and table before committing to the whole folder. Writes to the same output folder, so with `skip_existing = True` the batch won't redo it.

In [ ]:
test_file = Path(folder) / "PS8292.ims"

somas, log = soma.process_file(test_file, out_dir, params)
print(log)
display(Image(filename=str(out_dir / f"{test_file.stem}_qc.png")))
somas

## Run the whole folder

About 1.5 min per 3D stack. Progress prints one line per file. If it's interrupted, just rerun this cell: with `skip_existing = True` it picks up where it stopped.

In [ ]:
all_somas, run_log = soma.process_folder(folder, out_dir, params, pattern=pattern, skip_existing=skip_existing)
run_log

## Files to look at

Errors, skipped files, and stacks where no soma was kept. For a stack with no somas, its `_candidates.csv` shows which filter rejected each candidate.

In [ ]:
problems = run_log[(run_log["status"].isin(["error", "skipped"])) | (run_log["n_somas"] == 0)]
print(f"{len(problems)} file(s) to check")
problems[[c for c in ("file", "status", "n_somas", "n_candidates", "message") if c in problems]]

## Soma size summary

Somas touching the edge of the stack are cut off and measure too small, so they're excluded here (they stay in `all_soma_stats.csv` with `touches_border = True`). Strain = the file name before the first `_` (e.g. `PS8292`). Medians are used because one merged pair of somas can skew a mean.

`equivalent_diameter_um` is the diameter of a sphere with the soma's volume. To compare with an expanded nucleus, compare *diameters* (your 1.5x is a length factor); volumes scale with its cube (1.5³ ≈ 3.4x).

In [ ]:
all_somas = pd.read_csv(out_dir / "all_soma_stats.csv")
all_somas["strain"] = all_somas["file"].astype(str).str.split("_").str[0].str.replace(".ims", "", regex=False)
inside = all_somas[~all_somas["touches_border"].astype(bool)]
print(f"{len(all_somas)} soma(s) total, {len(all_somas) - len(inside)} touching the stack edge excluded")

if inside.empty:
    print("No somas to summarize: check run_log and the _candidates.csv / _qc.png files above.")
else:
    summary = inside.groupby("strain").agg(
        n_somas=("soma_id", "size"),
        n_files=("file", "nunique"),
        median_volume_um3=("volume_um3", "median"),
        median_diameter_um=("equivalent_diameter_um", "median"),
        q25_diameter_um=("equivalent_diameter_um", lambda s: s.quantile(0.25)),
        q75_diameter_um=("equivalent_diameter_um", lambda s: s.quantile(0.75)),
    ).round(2)
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(inside["equivalent_diameter_um"], bins=30)
    axes[0].set_xlabel("equivalent diameter (um)")
    axes[0].set_ylabel("somas")
    axes[0].set_title("All strains")
    inside.boxplot(column="equivalent_diameter_um", by="strain", ax=axes[1], rot=45)
    axes[1].set_ylabel("equivalent diameter (um)")
    axes[1].set_title("By strain")
    fig.suptitle("")
    plt.tight_layout()
    plt.show()